# Density Comparison — Papers vs Teams

Kernel-density estimate of each corpus over the shared joint-UMAP space, plotted
as `log2(teams_density / papers_density)`:

- **Red** = teams-dense relative to papers (iGEM activity ahead of the literature)
- **Blue** = papers-dense relative to teams (literature not yet adopted by iGEM)
- **White/neutral** = balanced coverage

This notebook also computes the **temporal precedence** of overlap-zone topics and
saves the tables consumed by `06-deliverables`.

> Run `compute_coords.ipynb` first.

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live in 05-reporting/ alongside the aux/ package.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

In [ ]:
import matplotlib.pyplot as plt

from aux.paths import REPORTS_DIR, set_seed
from aux.coords import load_plot_data
from aux.density import (
    compute_density_ratio, draw_density_heatmap, topic_zones,
    curated_density_labels, dfprec_extreme_labels,
)
from aux.labels import add_density_labels
from aux.precedence import compute_precedence, save_precedence

set_seed()
df_papers, df_teams, XLIM, YLIM = load_plot_data()
papers_xy = df_papers[["x", "y"]].to_numpy()
teams_xy = df_teams[["x", "y"]].to_numpy()
print(f"Papers: {len(df_papers):,} | Teams: {len(df_teams):,}")

## 1. Density-ratio grid

In [ ]:
dens = compute_density_ratio(papers_xy, teams_xy, XLIM, YLIM, grid_res=300, bw=0.15)
print(f"Grid {dens['grid_res']}x{dens['grid_res']}, ratio clipped to ±{dens['vmax']:.2f}")

## 2. Density-ratio heatmap (curated labels)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12), dpi=150)
draw_density_heatmap(ax, dens, XLIM, YLIM)

papers_ctr = topic_zones(df_papers, "topic", "low_name", dens["ratio_interp"])
teams_ctr = topic_zones(df_teams, "topic", "low_name", dens["ratio_interp"])
add_density_labels(ax, curated_density_labels(papers_ctr, teams_ctr, n=10), XLIM, YLIM)

ax.set_title("Density Ratio — Teams vs Papers (joint UMAP space)", fontsize=14)
fig.savefig(REPORTS_DIR / "umap_density_ratio.png", dpi=220, bbox_inches="tight")
plt.show()

## 3. Temporal precedence of overlap topics

For each overlap-zone paper topic, find the teams nearby in the joint space and
compare year distributions. `delta_*_years < 0` → iGEM teams preceded the
literature; `> 0` → the literature preceded iGEM.

In [ ]:
df_prec, papers_ctr, teams_ctr = compute_precedence(df_papers, df_teams, dens["ratio_interp"])
igem, lit = save_precedence(df_prec)

print("Papers topics by zone:", papers_ctr["zone"].value_counts().to_dict())
print(f"Overlap topics: {len(df_prec)} | iGEM-first: {len(igem)} | literature-first: {len(lit)}")
print(f"Saved → {REPORTS_DIR}: overlap_precedence_full.tsv, igem_preceded.tsv, literature_preceded.tsv")
df_prec.head(10)

## 4. Density-ratio heatmap (precedence extremes)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12), dpi=150)
draw_density_heatmap(ax, dens, XLIM, YLIM)
add_density_labels(ax, dfprec_extreme_labels(df_prec, papers_ctr, n_extreme=10), XLIM, YLIM)

ax.set_title("Density Ratio — precedence extremes (delta_q1_years)", fontsize=14)
fig.savefig(REPORTS_DIR / "umap_density_ratio_extremes.png", dpi=220, bbox_inches="tight")
plt.show()